# Monitoramento Inteligente: Porto de Santos
Este notebook marca o início do projeto de análise de tráfego marítimo no Porto de Santos. O objetivo é integrar modelos de detecção em tempo real com o poder analítico de modelos de linguagem (LLMs/VLMs).

## Objetivo do Projeto
Desenvolver um sistema capaz de monitorar streams de vídeo públicos para identificar embarcações e gerar relatórios automatizados de tráfego e ocorrências.

## Etapas de Desenvolvimento
Captura: Conexão com streams de câmeras online da área portuária.

Detecção (YOLO): Identificação e localização de embarcações em tempo real.

Análise (VLM): Classificação detalhada do porte e tipo do navio via visão computacional multimodal.

Relatório: Síntese diária de eventos e estatísticas de tráfego gerada por LLM.

## Notas de Exploração
Testar diferentes versões da YOLO para balanço entre latência e precisão.

Validar a qualidade dos crops enviados para a VLM.

Estruturar o armazenamento temporário das detecções para o fechamento do dia.

In [1]:
from ultralytics import YOLO

model = YOLO('../assets/yolo12m.pt')  # Carrega o modelo YOLOv8 nano para detecção de objetos

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\cacoc\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
import cv2
import yt_dlp

url = 'https://www.youtube.com/watch?v=tMYtrEBNVAU'

ydl_opts = {
    'format': 'best[ext=mp4][height<=480]',
    'quiet': True
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=False)
    video_url = info['url']

cap = cv2.VideoCapture(video_url)

seen = set()  # Conjunto para rastrear navios já detectados

while True:
    ret, frame = cap.read()
    
    if not ret:
        print("Não foi possível receber o frame (final do stream).")
        break

    # Detecção e rastreamento com YOLO apenas para barcos com confiança > 60%
    results = model.track(frame, conf=0.6, persist=True, verbose=False, show=False)
    for result in results:
        boxes = result.boxes
        for box in boxes:
            cls = int(box.cls)
            if result.names[cls] == 'boat':
                track_id = int(box.id) if box.id is not None else None
                if track_id is not None and track_id not in seen:
                    seen.add(track_id)
                    print(f"Novo navio detectado: ID {track_id}")  # Alerta único por navio
                # Desenhar a caixa
                x1, y1, x2, y2 = box.xyxy[0]
                conf = box.conf[0]
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0,255,0), 2)
                cv2.putText(frame, f'Boat {track_id} {conf:.2f}', (int(x1), int(y1)-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

    cv2.imshow('YouTube Stream', frame)

    # Altere de 1 para 66 para forçar a exibição a 15 FPS
    if cv2.waitKey(66) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()